In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
jawadali1045_20k_multi_class_crop_disease_images_path = kagglehub.dataset_download('jawadali1045/20k-multi-class-crop-disease-images')

print('Data source import complete.')


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow import keras
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


In [ ]:
tf.__version__,cv2.__version__

In [ ]:
!python --version

In [ ]:
print("Available devices:")
for device in tf.config.list_physical_devices():
    print(device)

# Data spliting into two folders

In [ ]:
import os

train_folder = 'datasets/jawadali1045/20k-multi-class-crop-disease-images/versions/1/Train'
val_folder = 'datasets/jawadali1045/20k-multi-class-crop-disease-images/versions/1/Validation'

# Get class names (same in Train and Validation)
class_names = sorted(os.listdir(train_folder))
print(f"Classes: {class_names}, Total: {len(class_names)}")

# Count images in Train and Validation
total_train_images = sum(len(os.listdir(os.path.join(train_folder, cls))) for cls in class_names if os.path.isdir(os.path.join(train_folder, cls)))
total_val_images = sum(len(os.listdir(os.path.join(val_folder, cls))) for cls in class_names if os.path.isdir(os.path.join(val_folder, cls)))

print(f"Total images in Train: {total_train_images}")
print(f"Total images in Validation: {total_val_images}")
print(f"Total images in dataset: {total_train_images + total_val_images}")

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Paths
train_folder = 'datasets/jawadali1045/20k-multi-class-crop-disease-images/versions/1/Train'
val_folder = 'datasets/jawadali1045/20k-multi-class-crop-disease-images/versions/1/Validation'
new_train_folder = '/kaggle/working/NewTrain'
test_folder = '/kaggle/working/Test'

# Create NewTrain and Test folders
os.makedirs(new_train_folder, exist_ok=True)
os.makedirs(test_folder, exist_ok=True)

# Set the test split ratio (20% for Test)
test_ratio = 0.20

# Iterate over each class in the Train folder
for class_name in os.listdir(train_folder):
    class_path = os.path.join(train_folder, class_name)

    if not os.path.isdir(class_path):
        continue

    # Get all image file names in the current class folder
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

    # Split images into Train and Test
    train_imgs, test_imgs = train_test_split(images, test_size=test_ratio, random_state=42)

    # Create class folders in NewTrain and Test
    new_train_class_folder = os.path.join(new_train_folder, class_name)
    test_class_folder = os.path.join(test_folder, class_name)
    os.makedirs(new_train_class_folder, exist_ok=True)
    os.makedirs(test_class_folder, exist_ok=True)

    # Copy remaining Train images to NewTrain
    for img in train_imgs:
        src = os.path.join(class_path, img)
        dst = os.path.join(new_train_class_folder, img)
        shutil.copy2(src, dst)

    # Copy Test images to Test
    for img in test_imgs:
        src = os.path.join(class_path, img)
        dst = os.path.join(test_class_folder, img)
        shutil.copy2(src, dst)

print("NewTrain and Test folders created successfully.")

# Verify the split
total_new_train_images = sum(len(os.listdir(os.path.join(new_train_folder, cls))) for cls in os.listdir(new_train_folder) if os.path.isdir(os.path.join(new_train_folder, cls)))
total_test_images = sum(len(os.listdir(os.path.join(test_folder, cls))) for cls in os.listdir(test_folder) if os.path.isdir(os.path.join(test_folder, cls)))
print(f"Total images in NewTrain: {total_new_train_images}")
print(f"Total images in Test: {total_test_images}")
print(f"Total images in Validation: {total_val_images}")
print(f"Total images across all sets: {total_new_train_images + total_val_images + total_test_images}")

# Data Visualization

In [ ]:
# Get all image paths
images = []
for folder in os.listdir(new_train_folder):
    path = os.path.join(new_train_folder,folder)
    print(path)
    for img in os.listdir(path):
        if img.endswith(('.jpg', '.png', '.webp')):
            images.append(os.path.join(path, img))

# Pick 16 random images
random_images = random.sample(images, 16)

# Plot them
plt.figure(figsize=(12, 12))
for i, img_path in enumerate(tqdm(random_images)):
    img = cv2.imread(img_path)
    if img is None:
        continue
    img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    plt.subplot(4, 4, i + 1)
    plt.imshow(img,cmap='gray')
    plt.axis('off')
    plt.title(f'Image {i+1}')

plt.tight_layout()
plt.show()

# CNN Model